# EA2 Unidad 1: Modelos de predicción y clasificación

Este notebook ejecuta el flujo completo de la actividad: carga automática desde Hugging Face, preprocesamiento, regresión lineal, clasificación, interpretación y generación de `resumen_modelos.txt`.

In [1]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATASET_ID = "scikit-learn/auto-mpg"
DATASET_URL = f"https://huggingface.co/datasets/{DATASET_ID}"
SEED = 42


/workspaces/ML_2026Q4_2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Carga automática y construcción del DataFrame

In [2]:
raw_dataset = load_dataset(DATASET_ID, split="train")
df_raw = raw_dataset.to_pandas()
print(f"Dataset: {DATASET_ID}")
print(f"Split: train")
print(f"Dimensiones: {df_raw.shape}")
print(f"Columnas: {df_raw.columns.tolist()}")
print("Tipos de datos:")
print(df_raw.dtypes.to_string())
print("Muestra:")
display(df_raw.head())


Dataset: scikit-learn/auto-mpg
Split: train
Dimensiones: (398, 9)
Columnas: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin', 'car name']
Tipos de datos:
mpg             float64
cylinders         int64
displacement    float64
horsepower       object
weight            int64
acceleration    float64
model year        int64
origin            int64
car name         object
Muestra:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## 1. Preprocesamiento y división de datos

In [3]:
df = df_raw.replace("?", np.nan).copy()
missing_before = df.isna().sum()
duplicates_before = int(df.duplicated().sum())
df = df.drop(columns=[column for column in ["car_name", "car name", "name"] if column in df])
df = df.drop_duplicates().copy()
median_mpg = float(df["mpg"].median())
df["efficiency"] = np.where(df["mpg"] >= median_mpg, "High", "Low")

X = df.drop(columns=["mpg", "efficiency"]).apply(pd.to_numeric, errors="coerce")
y_reg = df["mpg"]
y_cls = df["efficiency"]
print(f"Duplicados antes de limpiar: {duplicates_before}")
print("Nulos antes de imputar:")
print(missing_before[missing_before > 0].to_string() or "Ninguno")
print(f"Dimensiones después de limpiar: {df.shape}")
print("Distribución de clases:")
print(y_cls.value_counts().sort_index().to_string())

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=SEED
)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=SEED, stratify=y_cls
)
imputer_r = SimpleImputer(strategy="median")
imputer_c = SimpleImputer(strategy="median")
scaler_r = StandardScaler()
scaler_c = StandardScaler()
X_train_r_prepared = scaler_r.fit_transform(imputer_r.fit_transform(X_train_r))
X_test_r_prepared = scaler_r.transform(imputer_r.transform(X_test_r))
X_train_c_prepared = scaler_c.fit_transform(imputer_c.fit_transform(X_train_c))
X_test_c_prepared = scaler_c.transform(imputer_c.transform(X_test_c))
print(f"Regresión: train={len(X_train_r)}, test={len(X_test_r)}")
print(f"Clasificación: train={len(X_train_c)}, test={len(X_test_c)}")
print(f"Nulos después de imputar: {int(np.isnan(X_train_r_prepared).sum() + np.isnan(X_test_r_prepared).sum())}")


Duplicados antes de limpiar: 0
Nulos antes de imputar:
horsepower    6
Dimensiones después de limpiar: (398, 9)
Distribución de clases:
efficiency
High    201
Low     197
Regresión: train=318, test=80
Clasificación: train=318, test=80
Nulos después de imputar: 0


## 2. Regresión lineal

In [4]:
regression_model = LinearRegression()
regression_model.fit(X_train_r_prepared, y_train_r)
regression_predictions = regression_model.predict(X_test_r_prepared)
mae = mean_absolute_error(y_test_r, regression_predictions)
rmse = float(np.sqrt(mean_squared_error(y_test_r, regression_predictions)))
r2 = r2_score(y_test_r, regression_predictions)
coefficients = pd.Series(regression_model.coef_, index=X.columns)
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print("Coeficientes:")
print(coefficients.sort_values(ascending=False).to_string())
print(f"Intercepto: {regression_model.intercept_:.4f}")


MAE: 2.2556
RMSE: 2.8632
R²: 0.8475
Coeficientes:
model year      2.859148
displacement    1.465106
origin          1.073207
acceleration    0.187582
cylinders      -0.264273
horsepower     -0.494947
weight         -5.659110
Intercepto: 23.6082


## 3. Clasificación

In [5]:
classification_model = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=SEED
)
classification_model.fit(X_train_c_prepared, y_train_c)
classification_predictions = classification_model.predict(X_test_c_prepared)
accuracy = accuracy_score(y_test_c, classification_predictions)
precision = precision_score(y_test_c, classification_predictions, pos_label="High")
recall = recall_score(y_test_c, classification_predictions, pos_label="High")
f1 = f1_score(y_test_c, classification_predictions, pos_label="High")
labels = ["Low", "High"]
confusion = confusion_matrix(y_test_c, classification_predictions, labels=labels)
class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
    y_test_c, classification_predictions, labels=labels, zero_division=0
)
class_metrics = pd.DataFrame(
    {"precision": class_precision, "recall": class_recall, "f1": class_f1, "support": class_support},
    index=labels,
)
print(f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
display(class_metrics)
print("Matriz de confusión (filas reales, columnas predichas; [Low, High]):")
print(confusion)


Accuracy: 0.9750 | Precision: 0.9750 | Recall: 0.9750 | F1: 0.9750


,precision,recall,f1,support
Low,0.975,0.975,0.975,40
High,0.975,0.975,0.975,40


Matriz de confusión (filas reales, columnas predichas; [Low, High]):
[[39  1]
 [ 1 39]]


## 4. Interpretación y resumen automático

In [6]:
negative_feature = coefficients.idxmin()
positive_feature = coefficients.idxmax()
summary = f'''RESUMEN AUTOMÁTICO DE MODELOS DE MACHINE LEARNING
============================================================
Fecha de ejecución UTC: {datetime.now(timezone.utc).isoformat()}
Dataset: {DATASET_ID}
URL: {DATASET_URL}
Split: train
Dimensiones iniciales: {df_raw.shape}
Dimensiones después de limpiar: {df.shape}

PREPROCESAMIENTO
----------------
Nulos iniciales: {missing_before[missing_before > 0].to_dict()}
Duplicados iniciales: {duplicates_before}
Se reemplazaron '?' por NaN, se eliminaron nombres de vehículos y duplicados.
Se convirtieron los predictores a numéricos y se imputó la mediana.
Imputadores y escaladores se ajustaron únicamente con train.
La etiqueta efficiency usa la mediana de mpg ({median_mpg:.2f}): High >= mediana, Low < mediana.
Predictoras: {list(X.columns)}
División: 80% train / 20% test; random_state={SEED}.

REGRESIÓN LINEAL
----------------
Objetivo: predecir mpg a partir de características físicas y mecánicas.
Algoritmo: LinearRegression con parámetros predeterminados.
MAE: {mae:.4f}
RMSE: {rmse:.4f}
R2: {r2:.4f}
Intercepto: {regression_model.intercept_:.4f}
Coeficientes sobre variables estandarizadas:
{coefficients.to_string()}
Interpretación: el modelo explica {r2 * 100:.2f}% de la varianza de mpg.
El coeficiente más negativo es {negative_feature}; el más positivo es {positive_feature}.
Cada coeficiente representa el cambio en mpg ante una desviación estándar adicional,
manteniendo las demás variables constantes. El intercepto corresponde a predictores promedio.
Limitación: una relación lineal puede no capturar interacciones o no linealidades.

CLASIFICACIÓN
-------------
Objetivo: clasificar efficiency como High o Low.
Algoritmo: RandomForestClassifier(n_estimators=100, max_depth=5, random_state={SEED}).
Accuracy: {accuracy:.4f}
Precision High: {precision:.4f}
Recall High: {recall:.4f}
F1 High: {f1:.4f}
Matriz de confusión [Low, High], filas reales y columnas predichas:
{confusion}
Interpretación: aciertos Low={confusion[0, 0]}, aciertos High={confusion[1, 1]},
Low predicho como High={confusion[0, 1]}, High predicho como Low={confusion[1, 0]}.
Limitación: el resultado depende del umbral de la mediana usado para crear la etiqueta.

CONCLUSIÓN
----------
Ambos modelos se evaluaron sobre datos de prueba no vistos durante el ajuste.
Los resultados no implican causalidad y podrían cambiar con otros datos o particiones.
'''
with open("resumen_modelos.txt", "w", encoding="utf-8") as summary_file:
    summary_file.write(summary)
print(summary)
print("Archivo generado: resumen_modelos.txt")


RESUMEN AUTOMÁTICO DE MODELOS DE MACHINE LEARNING
Fecha de ejecución UTC: 2026-10-25T15:03:47.944630+00:00
Dataset: scikit-learn/auto-mpg
URL: https://huggingface.co/datasets/scikit-learn/auto-mpg
Split: train
Dimensiones iniciales: (398, 9)
Dimensiones después de limpiar: (398, 9)

PREPROCESAMIENTO
----------------
Nulos iniciales: {'horsepower': 6}
Duplicados iniciales: 0
Se reemplazaron '?' por NaN, se eliminaron nombres de vehículos y duplicados.
Se convirtieron los predictores a numéricos y se imputó la mediana.
Imputadores y escaladores se ajustaron únicamente con train.
La etiqueta efficiency usa la mediana de mpg (23.00): High >= mediana, Low < mediana.
Predictoras: ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin']
División: 80% train / 20% test; random_state=42.

REGRESIÓN LINEAL
----------------
Objetivo: predecir mpg a partir de características físicas y mecánicas.
Algoritmo: LinearRegression con parámetros predeterminados.
MAE: 2.